In [21]:
from preprocessing import dataset_preprocessing 

from preprocessing import detect_categories 

from tree import DecisionTree 

from sklearn.model_selection import train_test_split

import pandas as pd

import numpy as np

def training_split(dataset) : 

    train_data , test_data = train_test_split(dataset , test_size = 0.2 , random_state = 42 , stratify = dataset['satisfaction'])     
    return train_data , test_data

def tuning_split(train_data) :
    
    tuning_train , tuning_validation = train_test_split(train_data , test_size = 0.2 , random_state = 42 , stratify = train_data['satisfaction'])
    return tuning_train , tuning_validation 


def coarse_search(tuning_train , feats_dict , tuning_validation , n_iter , sample_size):

    coarse_search_df = {"max_depth" : [] , "min_samples_split" : [] ,"soft_max_leaf_nodes" : [] ,"soft_min_samples_leaf" : [] , "criterion" : [] ,
                         "train_F1-Score" : [] , "train_Acc" : [] , "validation_F1-Score" : [] , "validation_Acc" : []}
    _ , coarse_sample = train_test_split(tuning_train , test_size = (sample_size) / tuning_train.shape[0] , random_state = 42 , stratify = tuning_train['satisfaction'])

    search_domain = { "max_depth" : [None] + [i for i in range(5,30,2)] , "min_samples_split" : [2,5,10,15,20] ,
                      "soft_min_samples_leaf" : [1,2,5,10,15] , "soft_max_leaf_nodes" : [None] + [i for i in range(20,150,20)] ,
                      "criterion" : ['gini','gain'] }
    hyper_params = {}

    for _ in range(n_iter) :
        
        for hyper_param in list(coarse_search_df.keys())[:-4]:
           hyper_params[hyper_param] = np.random.choice(search_domain[hyper_param])
           coarse_search_df[hyper_param].append(hyper_params[hyper_param])
        
        decision_tree = DecisionTree()
        decision_tree.training(coarse_sample , hyper_params ,feats_dict.copy())
        
        F1_Score , Acc = decision_tree.tree_evaluation(coarse_sample)
        coarse_search_df["train_F1-Score"].append(F1_Score)
        coarse_search_df["train_Acc"].append(Acc)
        
        F1_Score , Acc = decision_tree.tree_evaluation(tuning_validation)
        coarse_search_df["validation_F1-Score"].append(F1_Score)
        coarse_search_df["validation_Acc"].append(Acc)

    return pd.DataFrame(coarse_search_df)
 
def fine_search(tuning_train , feats_dict , tuning_validation , coarse_search_df ,  n_grid) :

    fine_search_df = {"max_depth" : [] , "min_samples_split" : [] ,"soft_max_leaf_nodes" : [] ,"soft_min_samples_leaf" : [] , "criterion" : [] ,
                      "train_F1-Score" : [] , "train_Acc" : [] , "validation_F1-Score" : [] ,"validation_Acc" : []}
    coarse_search_df["weighted_score"] = 0.7 * coarse_search_df["validation_Acc"] + 0.3 * coarse_search_df["validation_F1-Score"]
    coarse_search_df.to_csv("../data/coarse_search.csv" , index = False)
    coarse_search_df = coarse_search_df.sort_values(by = "weighted_score" , ascending = False )
    hyper_params = {}

    for i in range(n_grid):
        
        for hyper_param in list(fine_search_df.keys())[:-4] :
            hyper_params[hyper_param] = coarse_search_df.loc[i , hyper_param]
            fine_search_df[hyper_param].append(hyper_params[hyper_param])
        
        decision_tree = DecisionTree()
        decision_tree.training(tuning_train , hyper_params , feats_dict.copy())
        
        F1_Score , Acc = decision_tree.tree_evaluation(tuning_train)
        fine_search_df["train_F1-Score"].append(F1_Score)
        fine_search_df["train_Acc"].append(Acc)
        
        F1_Score , Acc = decision_tree.tree_evaluation(tuning_validation)
        fine_search_df["validation_F1-Score"].append(F1_Score)
        fine_search_df["validation_Acc"].append(Acc)
        
        # pruning 

    fine_search_df = pd.DataFrame(fine_search_df)
    fine_search_df["weighted_score"] = 0.7 * fine_search_df["validation_Acc"] + 0.3 * fine_search_df["validation_F1-Score"]
    optimal_row = fine_search_df["weighted_score"].idxmax()
    fine_search_df.to_csv("../data/fine_search.csv" , index = False )

    for hyper_param in hyper_params.keys():
        hyper_params[hyper_param] = fine_search_df.loc[optimal_row , hyper_param]
    
    return hyper_params
    
def hyperparameter_tuning(train_data , feats_dict , n_iter , sample_size , n_grid):

    del feats_dict['satisfaction']
    tuning_train , tuning_validation = tuning_split(train_data)
    coarse_search_df = coarse_search(tuning_train , feats_dict , tuning_validation , n_iter , sample_size)
    return fine_search(tuning_train , feats_dict , tuning_validation , coarse_search_df , n_grid)

def training(train_data , test_data , optimal_hyper_params , feats_dict):

    del feats_dict['satisfaction']
    decision_tree = DecisionTree()
    decision_tree.training(train_data , optimal_hyper_params , feats_dict.copy())
    train_F1_score  , train_acc = decision_tree.tree_evaluation(train_data)
    test_F1_Score , test_acc = decision_tree.tree_evaluation(test_data)
    
    # pruning 

    return decision_tree , (train_F1_score , train_acc) , (test_F1_Score , test_acc)


In [22]:
# Decision_tree 1 trained on original dataset 

processed_dataset , original_column_categories = dataset_preprocessing(["id"],["Arrival Delay in Minutes"],["Arrival Delay in Minutes","Departure Delay in Minutes"],
{"Flight Distance" : 4 , "Arrival Delay in Minutes" : 4 , "Departure Delay in Minutes" : 4},{"Age" : 4},{})

# updated_columns_categories  

feats_dict = detect_categories(processed_dataset)

train_data , test_data = training_split(processed_dataset)

optimal_hyper_params = hyperparameter_tuning(train_data ,feats_dict.copy() , 60 , 32000 , 7)

decision_tree , train_metrics , test_metrics = training(train_data , test_data , optimal_hyper_params , feats_dict.copy())

print(f"train F1 score / train accuracy : {train_metrics}")

print(f"test F1 score / test accuracy : {test_metrics}")

/Users/faraz/university/second_term_4032/AP/DecisionTree/src/tree.py:159: RuntimeWarning: invalid value encountered in scalar divide
  return TP / (TP + FP)
/Users/faraz/university/second_term_4032/AP/DecisionTree/src/tree.py:159: RuntimeWarning: invalid value encountered in scalar divide
  return TP / (TP + FP)


train F1 score / train accuracy : (np.float64(0.3852061000015513), np.float64(0.4071195697941605))
test F1 score / test accuracy : (np.float64(0.15019326339039207), np.float64(0.1053847264327992))


In [23]:
# Decision_tree 2 trained on feature engineered dataset 

processed_dataset , original_column_categories = dataset_preprocessing(["id","Gender","Arrival Delay in Minutes","Departure Delay in Minutes","Departure/Arrival time convenient","Gate location"],[],
[],{"Flight Distance" : 4 },{"Age" : 4},{("Ease of Online booking","Inflight wifi service","Online boarding") : ("PCA","Online services") , 
("Seat comfort","Inflight entertainment","Cleanliness") : ("Mean","Comfort amenities") })

# updated_columns_categories  

feats_dict = detect_categories(processed_dataset)

train_data , test_data = training_split(processed_dataset)

optimal_hyper_params = hyperparameter_tuning(train_data ,feats_dict.copy() , 60 , 32000 , 7)

decision_tree , train_metrics , test_metrics = training(train_data , test_data , optimal_hyper_params , feats_dict.copy())

print(f"train F1 score / train accuracy : {train_metrics}")

print(f"test F1 score / test accuracy : {test_metrics}")

train F1 score / train accuracy : (np.float64(0.37243510307868105), np.float64(0.4088038208438098))
test F1 score / test accuracy : (np.float64(0.14224888716856976), np.float64(0.10384485828400943))
